In [1]:
from langchain_google_genai import ChatGoogleGenerativeAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from langchain_core.runnables import RunnablePassthrough,RunnableParallel
from langchain_community.document_loaders import PyPDFLoader,TextLoader,CSVLoader
import os

C:\Users\vigne\AppData\Local\Temp\ipykernel_18368\4083292498.py:5: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader,TextLoader,CSVLoader
C:\Users\vigne\AppData\Roaming\Python\Python314\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
from langchain_ollama import ChatOllama

In [3]:
parser = StrOutputParser()

In [4]:
model = ChatOllama(
    model="llama3.1"
)

In [5]:
sql_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an expert SQL query generator.

            Your task is to convert the user's natural language
            question into a correct SQL query.

            Use only the tables and columns provided in the
            database schema.

            Do not invent any tables or columns.
            Return only the SQL query.
            """
        ),
        (
            "human",
            """
            Database Schema:
            {schema}

            User Question:
            {question}

            Generate the SQL query.
            """
        )
    ]
)
sql_chain = sql_prompt | model | parser

In [6]:
explanation_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an SQL teacher.

            Explain the given SQL query in simple and
            beginner-friendly language.

            Explain what the query does and how it works.
            Do not modify the SQL query.
            """
        ),
        (
            "human",
            """
            Database Schema:
            {schema}

            User Question:
            {question}

            SQL Query:
            {sql_query}

            Explain this SQL query step by step.
            """
        )
    ]
)
explanation_chain = explanation_prompt | model | parser

In [7]:
tables_prompt = ChatPromptTemplate.from_messages(
    [
        (
            "system",
            """
            You are an SQL schema analyzer.

            Identify the tables and columns used in the
            given SQL query.

            Use only the tables and columns available
            in the database schema.
            """
        ),
        (
            "human",
            """
            Database Schema:
            {schema}

            SQL Query:
            {sql_query}

            Give the output in this format:

            Tables Used:
            - table_name

            Columns Used:
            - column_name
            """
        )
    ]
)
tables_chain = tables_prompt | model | parser

In [8]:
# final_chain = RunnableParallel(
#     sql = sql_chain,
#     explanation = explanation_chain,
#     tables = tables_chain
# )

In [11]:
import re
path = r"D:\Code\AI\Gen-AI\sales_data.csv"
loader = CSVLoader(path)
content = loader.load()
text = "\n".join(doc.page_content for doc in content)


full_pipeline = (
    RunnablePassthrough.assign(
        sql_query=sql_chain
    )
) | RunnablePassthrough.assign(
    explanation = explanation_chain,
    tables = tables_chain
)

while True:
    user_input = input("YOU: ")
    response = full_pipeline.invoke({
        'schema':text,
        'question':user_input
    })
    print(response)

# sql_query = sql_chain.invoke({
#     "schema": text,
#     "question": "Find the top 2 sales"
# })

# explanation = explanation_chain.invoke({
#     "schema": text,
#     "question": "Find the top 2 sales",
#     "sql_query": sql_query
# })

# tables = tables_chain.invoke({
#     "schema": text,
#     "sql_query": sql_query
# })



RuntimeError: Error loading D:\Code\AI\Gen-AI\sales_data.csv

In [ ]:
# print("SQL Query:")
# print(sql_query)

# print("\nExplanation:")
# print(explanation)

# print("\nTables:")
# print(tables)

SQL Query:
SELECT product, SUM(amount) AS total_amount
FROM customersales
GROUP BY product
ORDER BY total_amount DESC
LIMIT 2;

Explanation:
Let's break down the SQL query step by step.

**Step 1: SELECT**
The query starts with `SELECT`, which tells the database to retrieve data from the database.

**Step 2: product and SUM(amount) AS total_amount**
The query selects two columns:

* `product`: This column will retrieve the product names from the database.
* `SUM(amount) AS total_amount`: This is an aggregate function that calculates the total amount for each product. The `SUM()` function adds up all the amounts for each product. The `AS` keyword is used to give an alias to the calculated column, which is named `total_amount`.

**Step 3: FROM customersales**
The query specifies the table from which to retrieve data, which is `customersales`.

**Step 4: GROUP BY product**
The query groups the data by the `product` column. This means that the database will calculate the total amount for e